## 2A.Filter count comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import importlib
import src.utils as utils
importlib.reload(utils)

from src.utils import load_and_split_data
from src.utils import train_and_evaluate
from src.utils import BaselineCNN
from src.utils import run_Experiment_C





In [ ]:
def build_model(f1, f2, f3, f4):
    model = models.Sequential([
        layers.Input(shape=(32,32,3)),

        layers.Conv2D(f1, (3,3), activation='relu', padding='same'),
        layers.Conv2D(f2, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(f3, (3,3), activation='relu', padding='same'),
        layers.Conv2D(f4, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
small_model = build_model(8, 8, 16, 16)
medium_model = build_model(32, 32, 64, 64)
large_model = build_model(64, 64, 128, 128)

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

In [ ]:
compile_model(small_model)
compile_model(medium_model)
compile_model(large_model)

In [ ]:
hist_s, acc_s, loss_s, time_s = train_and_evaluate(
    small_model,
    x_train_C, y_train,
    x_val_C, y_val,
    x_test_C, y_test
)

hist_m, acc_m, loss_m, time_m = train_and_evaluate(
    medium_model,
    x_train_C, y_train,
    x_val_C, y_val,
    x_test_C, y_test
)

hist_l, acc_l, loss_l, time_l = train_and_evaluate(
    large_model,
    x_train_C, y_train,
    x_val_C, y_val,
    x_test_C, y_test
)

In [ ]:
small_model.summary()
medium_model.summary()
large_model.summary()

In [ ]:
print("Model | Params | Train Acc | Val Acc | Test Acc | Time")

print("Small  |", small_model.count_params(), acc_s, hist_s.history['val_accuracy'][-1], acc_s, time_s)
print("Medium |", medium_model.count_params(), acc_m, hist_m.history['val_accuracy'][-1], acc_m, time_m)
print("Large  |", large_model.count_params(), acc_l, hist_l.history['val_accuracy'][-1], acc_l, time_l)

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(hist_s.history['val_accuracy'], label='Small')
plt.plot(hist_m.history['val_accuracy'], label='Medium')
plt.plot(hist_l.history['val_accuracy'], label='Large')

plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Filter Size Comparison')
plt.legend()
plt.grid()
plt.show()

In [ ]:
#Shallow (4 conv layers):
def build_shallow():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
#Medium (6 conv layers):
def build_medium():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),

        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
#Deep (8 conv layers):
def build_deep():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),

        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
models_list = [
    ("Shallow", build_shallow()),
    ("Medium", build_medium()),
    ("Deep", build_deep())
]

histories = []
results = []

In [ ]:
#excecute training and evaluation for each model
for name, model in models_list:
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\nTraining {name} model...")

    history, acc, loss, t = train_and_evaluate(
        model,
        x_train_C, y_train,
        x_val_C, y_val,
        x_test_C, y_test
    )

    histories.append((name, history))
    results.append((name, acc, loss, t))

In [ ]:
#display results
df = pd.DataFrame(results, columns=["Model", "Accuracy", "Loss", "Time"])
df

In [ ]:
#plot training and validation loss curves
for name, history in histories:
    plt.figure()
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f"{name} Model Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()